# Google Colab Notebook associated with the paper "Using the "20 Questions" Game to explore LLM stochasticity"
Author: Eugenio Tufino, University of Modena and Reggio Emilia

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](
https://colab.research.google.com/github/etufino/AI-20-questions-game/blob/main/20QuestionsGamev_HFcompatibleV1_0.ipynb
)



⚠️ Privacy and Usage Disclaimer

Educational Purpose: This notebook is intended exclusively for educational experimentation.

Terms of Service: By inputting your API Key, you are interacting with Google Gemini via your personal account. Your activity is governed by the Google Generative AI Terms of Service.

Data Privacy: If you are using the Free Tier, please note that Google retains the right to use interaction data to tune and improve their models.

Safety: Do not share personal, confidential, or sensitive data within the chat interface.

👉 Generate your API Key at: [Google AI Studio](https://aistudio.google.com/app/apikey)

In [2]:
# Cell 1: Install + Imports + API Key

!pip -q install -U google-genai

import os, re, time
from types import SimpleNamespace

from google import genai
from google.genai import types

# Colab Secrets 🔑
try:
    from google.colab import userdata
    GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")
except Exception:
    GEMINI_API_KEY = os.environ.get("GEMINI_API_KEY")

if not GEMINI_API_KEY:
    raise ValueError("Missing GEMINI_API_KEY. Add it in Colab Secrets (🔑).")

MAX_QUESTIONS = 10  # just 10 questions

MODEL_UI_TO_CODE = {
    "Gemini 2.5 Flash (Fast)": "gemini-2.5-flash",
    "Gemini 2.5 Pro (Reasoning)": "gemini-2.5-pro",
}

def model_code_from_ui(model_ui: str) -> str:
    return MODEL_UI_TO_CODE.get(model_ui, "gemini-2.5-flash")


# session minimal
session = SimpleNamespace(client=None)

def ensure_client(session):
    if session.client is None:
        session.client = genai.Client(api_key=GEMINI_API_KEY)


In [3]:
# Cell 2: Core equivalent to  app.py HF (prompt + validate + guess + restart/resample)

import re, time

SYSTEM_PROMPT = (
    "You are an AI playing 'Guess the Object' (20 questions style).\n"
    "CONTEXT: January 2026.\n"
    "LANGUAGE: English.\n\n"
    "OUTPUT FORMAT (STRICT):\n"
    "1) Start with 'Yes' or 'No'.\n"
    "2) Then add: '—' + ONE short neutral micro-specification (6-15 words).\n"
    "   IMPORTANT: the micro-specification MUST justify the Yes/No with respect to the user's question.\n"
    "   It must NOT add new properties unrelated to the question.\n\n"
    "GOOD MICRO-SPECS:\n"
    "- A brief justification that directly matches the asked property.\n"
    "BAD MICRO-SPECS:\n"
    "- mentioning the object's specific purpose unless the question is about that.\n"
    "- adding new attributes not required to justify the Yes/No.\n"
)

REVEAL_PROMPT = (
    "CONTEXT:2026.\n"
    "Task: Name the ONE object you were simulating.\n"
    "Respond ONLY with the name in English (no punctuation, no explanation)."
)

# =========================
# UTILS
# =========================
def norm_text(s: str) -> str:
    s = (s or "").lower().strip()
    s = re.sub(r"[^\w\sàèéìòù]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

def norm_obj(s: str) -> str:
    s = (s or "").strip().lower()
    s = re.sub(r"[^\w\sàèéìòù]", "", s)
    s = re.sub(r"\s+", " ", s).strip()
    s = re.sub(r"\b(the|a|an)\b", "", s).strip()
    s = re.sub(r"\s+", " ", s).strip()
    return s

def objects_different(a: str, b: str) -> bool:
    na, nb = norm_obj(a), norm_obj(b)
    if not na or not nb:
        return True
    return na != nb

def only_yes_no(answer: str) -> bool:
    a = (answer or "").strip().lower()
    return bool(re.fullmatch(r"(yes|no)\.?\!?\??", a))

def starts_yes(answer: str) -> bool:
    return (answer or "").strip().lower().startswith("yes")

def starts_no(answer: str) -> bool:
    return (answer or "").strip().lower().startswith("no")

def detect_guess(message: str):
    if not message:
        return None
    m = re.match(r"^\s*guess\s*:?\s*(.+?)\s*$", message.strip(), flags=re.IGNORECASE)
    return m.group(1).strip() if m else None

# =========================
# Questions validation
# =========================
RE_REVEAL = re.compile(r"\b(what\s+is|what[’']s|which\s+is|tell\s+me|explain|reveal|identity)\b", flags=re.IGNORECASE)

def is_double_question(raw: str) -> bool:
    s = f" {raw.lower()} "
    if " or " in s and not re.search(r"\bor\s+not\b", s):
        return True
    return " either " in s

def validate_yesno_question(raw: str):
    if not raw or not raw.strip():
        return False, "Empty question."
    if RE_REVEAL.search(raw):
        return False, "Looks like a reveal/explanation request, not a yes/no question."
    if is_double_question(raw):
        return False, "Double question (contains 'or/either'). Ask one at a time."
    stripped = raw.strip()
    if stripped.endswith("?"):
        return True, ""
    m = norm_text(stripped)
    starters = (
        "is ", "are ", "can ", "could ", "does ", "do ", "did ", "has ", "have ",
        "was ", "were ", "should ", "would ", "it ", "there ",
        "is a", "is an", "is in", "is made", "is for"
    )
    if any(m.startswith(s) for s in starters):
        return True, ""
    return False, "Doesn't look like a yes/no question. Add '?' or rephrase as a binary property."

# =========================
# CLIENT + RETRY
# =========================
def _should_retry_error(msg: str) -> bool:
    m = (msg or "").lower()
    retry_tokens = ["503", "unavailable", "overloaded", "resource exhausted", "rate limit", "429", "timeout", "temporarily"]
    no_retry_tokens = ["401", "403", "permission", "api key", "invalid key", "unauthenticated"]
    if any(t in m for t in no_retry_tokens):
        return False
    return any(t in m for t in retry_tokens)

def safe_generate_text(session, model_code: str, contents: str, temperature: float, max_retries: int = 3):
    ensure_client(session)
    last_err = None
    for attempt in range(max_retries):
        try:
            resp = session.client.models.generate_content(
                model=model_code,
                contents=contents,
                config=types.GenerateContentConfig(temperature=float(temperature)),
            )
            txt = (resp.text or "").strip()
            if not txt:
                return None, "Empty response from the model."
            return txt, None
        except Exception as e:
            last_err = e
            msg = str(e)
            if _should_retry_error(msg) and attempt < max_retries - 1:
                time.sleep(0.7 * (2 ** attempt))
                continue
            return None, msg
    return None, str(last_err) if last_err else "Unknown error."

def enforce_micro_spec_if_needed(session, answer: str, model_code: str) -> str:
    txt = (answer or "").strip()
    if not only_yes_no(txt):
        return txt

    rewrite_prompt = (
        "Rewrite the answer following the rule:\n"
        "- Start with 'Yes' or 'No'\n"
        "- THEN add '—' and ONE neutral micro-specification (6-15 words)\n"
        "- Forbidden: the object's name, direct synonyms, brands\n\n"
        f"Original answer: {txt}\n"
        "Corrected answer:"
    )
    new_txt, err = safe_generate_text(session, model_code, rewrite_prompt, temperature=0.2, max_retries=3)
    return (new_txt or txt).strip()

# =========================
# LLM CORE
# =========================
def llm_respond(session, log_to_use, user_question: str, temperature: float, model_ui: str):
    model_code = model_code_from_ui(model_ui)

    prompt_parts = [SYSTEM_PROMPT, "\n--- History ---"]
    if not log_to_use:
        prompt_parts.append("No previous questions.")
    else:
        for i, step in enumerate(log_to_use):
            prompt_parts.append(f"Q{i+1}: {step['q']} -> A: {step['a']}")
    prompt_parts.append(f"--- New Q ---\nUser: {user_question}\nAnswer:")

    txt, err = safe_generate_text(session, model_code, "\n".join(prompt_parts), temperature=temperature, max_retries=3)
    if err:
        return None, err
    txt = enforce_micro_spec_if_needed(session, txt, model_code)
    return txt, None

def llm_reveal(session, log_to_use, model_ui: str):
    model_code = model_code_from_ui(model_ui)
    hist = "\n".join([f"Q: {x['q']} A: {x['a']}" for x in log_to_use])
    prompt = f"{hist}\n\n{REVEAL_PROMPT}"
    txt, err = safe_generate_text(session, model_code, prompt, temperature=0.2, max_retries=3)
    if err:
        return None, err
    return txt.strip(), None

def llm_check_guess_consistency(session, log_to_use, guess: str, model_ui: str):
    model_code = model_code_from_ui(model_ui)
    hist = "\n".join([f"Q: {x['q']} A: {x['a']}" for x in log_to_use])
    prompt = (
        "You are a referee.\n"
        "Given the history of questions/answers (Yes/No) below, decide whether the proposed object is compatible with ALL answers.\n"
        "Answer EXACTLY with a single word: YES or NO.\n\n"
        f"PROPOSED OBJECT: {guess}\n\n"
        f"HISTORY:\n{hist}\n\n"
        "VERDICT (YES/NO):"
    )
    txt, err = safe_generate_text(session, model_code, prompt, temperature=0.2, max_retries=3)
    if err:
        return None, err

    verdict = (txt.strip().split() or [""])[0].lower()
    if verdict == "yes":
        return True, None
    if verdict == "no":
        return False, None
    return False, None

# =========================
# CLI GAME (HF-like: Line A / Restart from Qk / Line B + RESAMPLE COMMAND)
# =========================
def play_physis_cli(model_ui="Gemini 2.5 Flash (Fast)", temperature=1.0):
    internal_log = []   # Line A
    rewind_log = []     # Line B

    print("\nPhysis (CLI) — coherent with HF app.py")
    print(f"Model: {model_ui} | MAX_QUESTIONS={MAX_QUESTIONS}")
    print("Commands: Guess: <obj> | Finish | Restart | Resample [N] | r [N] | T=<value>")
    print("-" * 70)

    # -------- Phase 1 (Line A)
    print("\nPHASE 1 (Line A): ask yes/no questions.")
    original_object = None

    while len(internal_log) < MAX_QUESTIONS:
        prompt = f"[Line A | Q{len(internal_log)+1}/{MAX_QUESTIONS} | T={temperature}] > "
        msg = input(prompt).strip()
        if not msg:
            continue

        # temperature command
        mT = re.match(r"^\s*T\s*=\s*([0-9.]+)\s*$", msg, flags=re.IGNORECASE)
        if mT:
            temperature = float(mT.group(1))
            print(f"(Temperature set to {temperature})")
            continue

        if msg.lower() == "finish":
            if not internal_log:
                print("Ask at least 1 question before finishing.")
                continue
            break

        g = detect_guess(msg)
        if g is not None:
            g = norm_obj(g)
            if not g:
                print("Invalid guess. Example: Guess: mug")
                continue
            ok, err = llm_check_guess_consistency(session, internal_log, g, model_ui)
            if err:
                print(f"Model error (guess check): {err}")
                continue
            if ok:
                print(f"YES — '{g}' is compatible. (End Phase 1)")
                original_object = g
                break
            else:
                print("NO — not compatible.")
                continue

        okq, reason = validate_yesno_question(msg)
        if not okq:
            print(f"Invalid (doesn't count): {reason}")
            continue

        ans, err = llm_respond(session, internal_log, msg, temperature, model_ui)
        if err:
            print(f"Model error (not counted): {err}")
            continue

        print("LLM:", ans)
        internal_log.append({"q": msg, "a": ans})

    # reveal Line A if not guessed
    if original_object is None:
        obj, err = llm_reveal(session, internal_log, model_ui)
        original_object = obj if not err else "Unknown"
        if err:
            print(f"Reveal error: {err}")

    print("\nEND PHASE 1")
    print("Line A object:", original_object)

    # -------- Choose restart point
    while True:
        print("\nChoose Restart from Qk:")
        for i, step in enumerate(internal_log):
            print(f"{i+1}: Restart from Q{i+1}: {step['q']}")
        choice = input("Enter k (1..N) > ").strip()
        try:
            k = int(choice)
            if 1 <= k <= len(internal_log):
                break
        except ValueError:
            pass
        print("Invalid k.")

    pivot_index = k - 1
    pivot_question = internal_log[pivot_index]["q"]
    pivot_answer_A = internal_log[pivot_index]["a"]

    # Line B starts with history up to Q(k-1)
    rewind_log = internal_log[:pivot_index]

    # Resample state: current question is pivot_question (slot Qk)
    rerun_base_log = rewind_log[:]            # base history for resampling
    rerun_target_question = pivot_question    # current question to resample
    rerun_slot_index = len(rewind_log)        # where Qk lives in rewind_log
    rerun_samples = rerun_yes = rerun_no = 0

    print("\nRESTART → LINE B")
    print(f"Pivot Q{k} (Line A): {pivot_question}")
    print(f"Answer (Line A): {pivot_answer_A}")
    print("Now in Line B.")
    print("Use Resample/r to repeat the current question without advancing.")
    print("Type a NEW question to advance to the next slot.")
    print("-" * 70)

    # -------- Phase 3 (Line B)
    while len(rewind_log) < MAX_QUESTIONS:
        # HF-like prompt: show resample slot + next NEW slot
        resample_q = rerun_slot_index + 1
        next_new_q = len(rewind_log) + 1
        prompt = f"[Line B | resample Q{resample_q} | new Q{next_new_q}/{MAX_QUESTIONS} | T={temperature}] > "

        msg = input(prompt).strip()
        if not msg:
            continue

        # temperature command
        mT = re.match(r"^\s*T\s*=\s*([0-9.]+)\s*$", msg, flags=re.IGNORECASE)
        if mT:
            temperature = float(mT.group(1))
            print(f"(Temperature set to {temperature})")
            continue

        if msg.lower() == "restart":
            return play_physis_cli(model_ui=model_ui, temperature=temperature)

        if msg.lower() == "finish":
            if not rewind_log:
                print("Ask at least 1 question in Line B before finishing.")
                continue
            objB, err = llm_reveal(session, rewind_log, model_ui)
            if err:
                objB = "Unknown"
                print(f"Reveal error (Line B): {err}")
            print("\nEND LINE B")
            print("Line B object:", objB)
            changed = objects_different(original_object, objB)
            print("RESULT:", "OBJECT CHANGED (BIFURCATION)" if changed else "OBJECT STABLE")
            print("Line A:", original_object)
            print("Line B:", objB)
            return

        # RESAMPLE command: resample [N]  OR  r [N]
        mR = re.match(r"^\s*(resample|r)\s*(\d+)?\s*$", msg, flags=re.IGNORECASE)
        if mR:
            if not rerun_target_question:
                print("No current question to resample yet.")
                continue

            n = int(mR.group(2)) if mR.group(2) else 1
            for _ in range(n):
                ans, err = llm_respond(session, rerun_base_log, rerun_target_question, temperature, model_ui)
                if err:
                    print(f"Model error (not counted): {err}")
                    break

                rerun_samples += 1
                if starts_yes(ans): rerun_yes += 1
                elif starts_no(ans): rerun_no += 1

                i = rerun_slot_index
                # overwrite same slot (does not advance)
                if len(rewind_log) == i:
                    rewind_log.append({"q": rerun_target_question, "a": ans})
                else:
                    rewind_log[i]["q"] = rerun_target_question
                    rewind_log[i]["a"] = ans

                print(f"RESAMPLE #{rerun_samples} | Yes={rerun_yes} No={rerun_no}")
                print(f"Q{i+1}: {rerun_target_question}")
                print("LLM:", ans)
            continue

        # Guess (does not count)
        g = detect_guess(msg)
        if g is not None:
            g = norm_obj(g)
            if not g:
                print("Invalid guess. Example: Guess: mug")
                continue
            ok, err = llm_check_guess_consistency(session, rewind_log, g, model_ui)
            if err:
                print(f"Model error (guess check): {err}")
                continue
            if ok:
                objB = g
                print("\nEND LINE B (by Guess)")
                print("Line B object:", objB)
                changed = objects_different(original_object, objB)
                print("RESULT:", "OBJECT CHANGED (BIFURCATION)" if changed else "OBJECT STABLE")
                print("Line A:", original_object)
                print("Line B:", objB)
                return
            else:
                print("NO — not compatible.")
                continue

        # Validate yes/no question (counts only if valid + model returns)
        okq, reason = validate_yesno_question(msg)
        if not okq:
            print(f"Invalid (doesn't count): {reason}")
            continue

        ans, err = llm_respond(session, rewind_log, msg, temperature, model_ui)
        if err:
            print(f"Model error (not counted): {err}")
            continue

        print("LLM:", ans)

        # NORMAL question ADVANCES:
        # It fills the next available slot (append) and becomes the new current resample question.
        rewind_log.append({"q": msg, "a": ans})
        rerun_base_log = rewind_log[:-1]
        rerun_target_question = msg
        rerun_slot_index = len(rerun_base_log)
        rerun_samples = rerun_yes = rerun_no = 0

    # auto-finish if you hit MAX_QUESTIONS
    objB, err = llm_reveal(session, rewind_log, model_ui)
    if err:
        objB = "Unknown"
        print(f"Reveal error (Line B): {err}")
    print("\nEND LINE B (max questions reached)")
    print("Line B object:", objB)
    changed = objects_different(original_object, objB)
    print("RESULT:", "OBJECT CHANGED (BIFURCATION)" if changed else "OBJECT STABLE")
    print("Line A:", original_object)
    print("Line B:", objB)


In [4]:
# Cell 3A: Settings (choose initial temperature)

T0 = 1.3  # change this value (e.g., 0.2, 0.7, 1.0, 1.3, 1.8)
T0 = max(0.0, min(2.0, float(T0)))
print(f"Initial temperature set to T0 = {T0}")



Initial temperature set to T0 = 1.3


In [5]:

# Cell 3B: Run
play_physis_cli(model_ui="Gemini 2.5 Flash (Fast)", temperature=T0)



Physis (CLI) — coherent with HF app.py
Model: Gemini 2.5 Flash (Fast) | MAX_QUESTIONS=10
Commands: Guess: <obj> | Finish | Restart | Resample [N] | r [N] | T=<value>
----------------------------------------------------------------------

PHASE 1 (Line A): ask yes/no questions.
[Line A | Q1/10 | T=1.3] > is it a living being?
LLM: No — it is an inanimate object.
[Line A | Q2/10 | T=1.3] > is it relatively large?
LLM: Yes — it is a structure of considerable height and width.
[Line A | Q3/10 | T=1.3] > is it movable?
LLM: No — it is built to remain permanently stationary.
[Line A | Q4/10 | T=1.3] > is it used near the river?
LLM: Yes — it is commonly constructed over or adjacent to water bodies.
[Line A | Q5/10 | T=1.3] >  Is it usually used for passing cars?
LLM: Yes — it is engineered to support the movement of vehicles.
[Line A | Q6/10 | T=1.3] > guess: bridge
YES — 'bridge' is compatible. (End Phase 1)

END PHASE 1
Line A object: bridge

Choose Restart from Qk:
1: Restart from Q1: is